<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

### 📊 Resumo de Metadados: NEX-GDDP-CMIP6 (GFDL-ESM4)

| Grandeza              | Nome Longo                                 | Nome Curto | Unidade    | Shape (Time, Lat, Lon) |
|:----------------------|:-------------------------------------------|:-----------|:-----------|:-----------------------|
| **Relative Humidity** | Near-Surface Relative Humidity             | `hurs`     | %          | [365, 600, 1440]       |
| **Precipitation**     | Precipitation                              | `pr`       | kg m-2 s-1 | [365, 600, 1440]       |
| **Solar Radiation**   | Surface Downwelling Shortwave Radiation    | `rsds`     | W m-2      | [365, 600, 1440]       |
| **Wind Speed**        | Daily-Mean Near-Surface Wind Speed         | `sfcWind`  | m s-1      | [365, 600, 1440]       |
| **Max Temperature**   | Daily Maximum Near-Surface Air Temperature | `tasmax`   | K          | [365, 600, 1440]       |
| **Min Temperature**   | Near-Surface Air Temperature (Daily Min)   | `tasmin`   | K          | [365, 600, 1440]       |

**Coordenadas Globais (Amostra 2015):**
*   **Latitude:** -59.88° a 89.88° (Tamanho: 600)
*   **Longitude:** 0.12° a 359.88° (Tamanho: 1440)

In [2]:
import os
import shutil
import glob
import gc
import re
import numpy as np
import dask
from dask.diagnostics import ProgressBar
import xarray as xr
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import geopandas as gpd
import rioxarray

# 1. DEFINIÇÃO DE PARÂMETROS E DIRETÓRIOS
dask.config.set(scheduler='single-threaded')

MODELOS = [#'GFDL-ESM4',
           #'IPSL-CM6A-LR',
           #'MPI-ESM1-2-HR',
           #'MRI-ESM2-0',
           'UKESM1-0-LL'
           ]
CENARIOS = ['ssp585']
VARIAVEIS = ['hurs', 'pr', 'rsds', 'sfcWind', 'tasmax', 'tasmin']

AWS_BUCKET_NAME = 'nex-gddp-cmip6'
BASE_PREFIX = 'NEX-GDDP-CMIP6'
ANO_INICIO = 2015
ANO_FIM = 2100

DIRETORIO_SAIDA_BASE = r'G:\Drives compartilhados\GAS-Henrique\NEX-GDDP-CMIP6'
TEMP_NC = r'D:\temp_nc'

s3_config = Config(
    signature_version=UNSIGNED,
    retries={'max_attempts': 5, 'mode': 'standard'},
    connect_timeout=10,
    read_timeout=30
)
s3 = boto3.client('s3', config=s3_config)
paginator = s3.get_paginator('list_objects_v2')

# 2. FUNÇÕES DE PROCESSAMENTO
def limpar_temp():
    if os.path.exists(TEMP_NC):
        shutil.rmtree(TEMP_NC, ignore_errors=True)
    os.makedirs(TEMP_NC, exist_ok=True)

def padronizar_dataset(ds):
    if 'pr' in ds:
        ds['pr'] = ds['pr'] * 86400
        ds['pr'].attrs['units'] = 'mm/day'
    if 'rsds' in ds:
        ds['rsds'] = ds['rsds'] * 0.0864
        ds['rsds'].attrs['units'] = 'MJ m-2 day-1'
    if 'tasmax' in ds:
        ds['tasmax'] = ds['tasmax'] - 273.15
        ds['tasmax'].attrs['units'] = 'degC'
    if 'tasmin' in ds:
        ds['tasmin'] = ds['tasmin'] - 273.15
        ds['tasmin'].attrs['units'] = 'degC'

    if 'lon' in ds.coords:
        ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
        ds = ds.sortby(ds.lon)
    if 'lat' in ds.coords:
        ds = ds.sortby(ds.lat)
    return ds

# --- MAPEAMENTO LOCAL PRÉVIO ---
print("Mapeando arquivos já existentes no Google Drive em memória...")
arquivos_processados = set()
if os.path.exists(DIRETORIO_SAIDA_BASE):
    for root, _, files in os.walk(DIRETORIO_SAIDA_BASE):
        for f in files:
            if f.endswith('.nc4'):
                arquivos_processados.add(f)
print(f"✅ {len(arquivos_processados)} arquivos ignorados por já estarem concluídos.\n")

# Carrega o shapefile atualizado
CAMINHO_SHAPEFILE = r'G:\Drives compartilhados\GAS-Henrique\shapefiles.shp'
print("Carregando limites continentais...")
gdf_continentes = gpd.read_file(CAMINHO_SHAPEFILE)

# 3. LOOP PRINCIPAL
try:
    for modelo in MODELOS:
        for cenario in CENARIOS:
            print(f"\n[{modelo} | {cenario}] Iniciando mapeamento no S3...")
            prefixo_busca = f"{BASE_PREFIX}/{modelo}/{cenario}/"

            try:
                for page in paginator.paginate(Bucket=AWS_BUCKET_NAME, Prefix=prefixo_busca):
                    if 'Contents' not in page: continue

                    for obj in page['Contents']:
                        key = obj['Key']
                        if not key.endswith('.nc'): continue

                        nome_arq = key.split('/')[-1]
                        match_var = re.match(r'^([a-zA-Z0-9]+)_', nome_arq)
                        match_ano = re.search(r'_(\d{4})(?:_v2\.0)?\.nc$', nome_arq)

                        if not match_var or not match_ano:
                            continue

                        var_arq = match_var.group(1)
                        ano_arq = int(match_ano.group(1))

                        if (var_arq in VARIAVEIS) and (ANO_INICIO <= ano_arq <= ANO_FIM):

                            nome_saida = f"{modelo}_{cenario}_{var_arq}_{ano_arq}.nc4"

                            # Checkpoint local ultrarrápido
                            if nome_saida in arquivos_processados:
                                continue

                            # Criação da estrutura de pastas sob demanda
                            pasta_saida = os.path.join(DIRETORIO_SAIDA_BASE, modelo, cenario, var_arq)
                            os.makedirs(pasta_saida, exist_ok=True)
                            arquivo_saida_drive = os.path.join(pasta_saida, nome_saida)

                            print(f"  -> Processando: {nome_saida}")
                            limpar_temp()
                            caminho_local_in = os.path.join(TEMP_NC, nome_arq)
                            caminho_local_out = os.path.join(TEMP_NC, nome_saida)

                            try:
                                # 1. Download
                                s3.download_file(AWS_BUCKET_NAME, key, caminho_local_in)

                                # 2. Abertura e Padronização de Calendário
                                # Nova sintaxe do Xarray usando CFDatetimeCoder para evitar Warnings
                                time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
                                ds = xr.open_dataset(caminho_local_in, decode_times=time_coder)

                                # Detecção inteligente de calendário (suporta várias versões do xarray)
                                if hasattr(ds.time.dt, 'calendar'):
                                    cal = ds.time.dt.calendar
                                else:
                                    cal = ds.time.encoding.get('calendar', 'standard')

                                # Lógica de conversão flexível
                                if cal == '360_day':
                                    # UKESM1 e outros britânicos: 360 dias.
                                    # align_on="random" espalha os dias e cria 5 dias vazios no ano
                                    ds = ds.convert_calendar("noleap", align_on="random")
                                    # Preenche as lacunas artificialmente para manter a consistência diária
                                    ds = ds.interpolate_na(dim='time', method='linear')
                                else:
                                    # Padrão ou já noleap: converte normalmente
                                    ds = ds.convert_calendar("noleap")

                                # Padroniza unidades
                                ds = padronizar_dataset(ds)

                                # 3. Recorte Espacial
                                ds.rio.write_crs("epsg:4326", inplace=True)
                                ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
                                ds = ds.rio.clip(gdf_continentes.geometry, gdf_continentes.crs, drop=True)

                                # 4. Força a compressão padrão para a variável ativa
                                if var_arq in ds.data_vars:
                                    ds[var_arq].encoding = {
                                        'zlib': True,
                                        'complevel': 5,
                                        '_FillValue': np.nan
                                    }

                                # 5. Salva localmente (VM)
                                ds.to_netcdf(caminho_local_out, engine='h5netcdf', format='NETCDF4')
                                ds.close()

                                # 6. Move com segurança para o Drive e atualiza cache
                                shutil.move(caminho_local_out, arquivo_saida_drive)
                                arquivos_processados.add(nome_saida)

                            except Exception as e:
                                print(f"     ❌ Erro em {nome_saida}: {e}")
                                if os.path.exists(caminho_local_out): os.remove(caminho_local_out)
                            finally:
                                if 'ds' in locals(): del ds
                                gc.collect()

            except Exception as e:
                print(f"    ❌ Falha de rede ao listar S3 ({modelo}/{cenario}): {e}")
                continue

except KeyboardInterrupt:
    print("\n⛔ Execução interrompida manualmente pelo usuário.")

limpar_temp()
print("\nProcessamento atomizado finalizado.")

Mapeando arquivos já existentes no Google Drive em memória...
✅ 9690 arquivos ignorados por já estarem concluídos.

Carregando limites continentais...

[UKESM1-0-LL | ssp585] Iniciando mapeamento no S3...

Processamento atomizado finalizado.
